In [ ]:
# Path to your .spindle.json or DDL file
SCHEMA_PATH   = "/path/to/your/schema.spindle.json"  # or .sql DDL file

SINK_TYPE     = "lakehouse"   # lakehouse | warehouse | sql-database | sql-server
SCALE_SIZE    = "small"
SEED          = 42

WORKSPACE_ID  = ""   # Fabric workspace GUID (sink=lakehouse)
LAKEHOUSE_ID  = ""   # Lakehouse GUID (sink=lakehouse)
SQL_CONN      = ""   # ODBC string (sink=warehouse|sql-database|sql-server)


In [ ]:
from pathlib import Path
from sqllocks_spindle import Spindle

path = Path(SCHEMA_PATH)
if not path.exists():
    raise FileNotFoundError(f"Schema file not found: {SCHEMA_PATH}")

if path.suffix == ".json":
    import json
    schema_def = json.loads(path.read_text())
    result = Spindle().generate(schema=schema_def, scale=SCALE_SIZE, seed=SEED)
elif path.suffix == ".sql":
    from sqllocks_spindle.schema.parser import DDLParser
    schema = DDLParser().parse(path.read_text())
    result = Spindle().generate(schema=schema, scale=SCALE_SIZE, seed=SEED)
else:
    raise ValueError(f"Unsupported schema file type: {path.suffix} (use .json or .sql)")

errors = result.verify_integrity()
assert errors == [], f"FK integrity: {errors}"
for t, df in result.tables.items():
    print(f"  {t}: {len(df):,} rows")


In [ ]:
if SINK_TYPE == "lakehouse":
    from azure.identity import InteractiveBrowserCredential
    from deltalake import write_deltalake
    token = InteractiveBrowserCredential(tenant_id="2536810f-20e1-4911-a453-4409fd96db8a").get_token("https://storage.azure.com/.default").token
    opts = {"bearer_token": token, "use_emulator": "false"}
    for name, df in result.tables.items():
        path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/custom_{name}"
        write_deltalake(path, df, mode="overwrite", storage_options=opts, schema_mode="overwrite")
        print(f"  custom_{name}: {len(df):,} rows -> lakehouse")
elif SINK_TYPE in ("warehouse", "sql-database", "sql-server"):
    from sqllocks_spindle.fabric.sql_database_writer import FabricSqlDatabaseWriter
    auth = "sql" if SINK_TYPE == "sql-server" else "cli"
    writer = FabricSqlDatabaseWriter(SQL_CONN, auth_method=auth)
    writer.write(result, schema_name="dbo", mode="create_insert")
    print(f"Written to {SINK_TYPE}")
print("Done")
